# ⚙️ 5. Before-Agent Guardrail

A **before-agent guardrail** runs before the agent starts processing the user's request.

It can inspect incoming input and block requests that violate predefined rules.

### Best for:
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

---

## 🔄 How Before-Agent Guardrails Work

```text
User Input
    ↓
Before-Agent Guardrail
    ↓
Input Allowed?
    ├── Yes → 🤖 Agent Executes
    └── No  → 🚫 Block Request

## Before-Agent Guardrail Example

We can create a custom middleware using before_agent to inspect user input before the agent executes.

This approach is deterministic because it uses predefined banned keywords.

### Import Required Libraries

In [1]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq

### Initialize Groq Model

In [2]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

### Create Content Filter Middleware

In [3]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        first_message = state["messages"][0]

        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:

            if keyword in content:

                print(f"🚫 Blocked — keyword detected: '{keyword}'")

                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }

        return None

### Create Search Tool

In [4]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

### Create Agent with Content Filter

In [5]:
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=[
                "hack",
                "exploit",
                "malware",
                "jailbreak",
                "bypass"
            ]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


### Test 1: Safe request — should pass through

In [6]:
result = filtered_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is machine learning?"
        }
    ]
})

print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
**Machine learning (ML)** is a subfield of artificial intelligence (AI) that focuses on building systems that can learn from data, identify patterns, and make decisions or predictions without being explicitly programmed for each specific task.

### Core Idea
- **Learning from data:** Instead of writing explicit rules, you feed a model a large set of examples (the *training data*). The model automatically discovers the underlying relationships.
- **Generalization:** After training, the model can apply what it has learned to new, unseen data (the *test* or *deployment* data).

### How It Works (High‑Level Steps)
1. **Collect data** – Gather relevant examples (e.g., images, text, sensor readings).
2. **Preprocess** – Clean, normalize, and possibly transform the data into a suitable format.
3. **Choose a model** – Select an algorithm (e.g., linear regression, decision tree, neural network) that can represent the mapping from inputs to outputs.
4. **Train the model*

### Test 2: Unsafe request — should be blocked

In [7]:
result = filtered_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How do I hack into a server?"
        }
    ]
})

print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


---

## 💡 Why Use a Before-Agent Guardrail?

The main advantage is that blocked requests can be stopped **before the agent performs any reasoning or tool calls**.

This makes deterministic input filtering:

- ⚡ Fast
- 💰 Cost-effective
- 🔒 Predictable
- 🛠️ Easy to debug

---

## 📝 Key Takeaways

- A **before-agent guardrail** runs before the agent processes a request.
- Custom middleware can inspect and filter user input.
- `before_agent` is used to implement the input check.
- `@hook_config(can_jump_to=["end"])` allows the middleware to stop the agent workflow.
- Deterministic keyword filtering does not require an additional LLM call.
- Before-agent guardrails are useful for **blocking unsafe or unwanted requests early**.